# Bloque III — Clasificación, matriz de confusión y ROC

**Duración estimada:** 3 horas  
**Dataset:** `../data/clientes_abandono_mayo_2026.csv`

## Objetivo de aprendizaje

El alumnado aprenderá a entrenar modelos de clasificación, interpretar sus métricas y ajustar decisiones en función del coste de falsos positivos y falsos negativos.

## Agenda de 3 horas

| Tiempo | Actividad |
|---:|---|
| 0:00–0:25 | Problemas de clasificación |
| 0:25–0:55 | Preparación de datos y balanceo |
| 0:55–1:25 | Regresión logística y árboles |
| 1:25–1:35 | Pausa |
| 1:35–2:10 | Accuracy, precision, recall y F1 |
| 2:10–2:35 | Matriz de confusión y ROC |
| 2:35–3:00 | Caso práctico |

In [ ]:
# Configuración común
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score,
    classification_report
)

## 1. Carga del dataset

El objetivo será predecir si un cliente abandonará (`abandono = 1`) o no (`abandono = 0`).

In [ ]:
df = pd.read_csv("../data/clientes_abandono_mayo_2026.csv")
df.head()

## 2. Distribución de clases

En clasificación es fundamental revisar si las clases están balanceadas. Un dataset desequilibrado puede hacer que `accuracy` sea engañosa.

In [ ]:
conteo = df["abandono"].value_counts()
proporcion = df["abandono"].value_counts(normalize=True)

display(conteo)
display(proporcion)

In [ ]:
plt.figure(figsize=(5, 4))
conteo.plot(kind="bar")
plt.title("Distribución de la variable objetivo")
plt.xlabel("Abandono")
plt.ylabel("Número de clientes")
plt.show()

## 3. Preparación de X e y

Separamos variables predictoras y variable objetivo. Usamos estratificación para conservar la proporción de clases en train y test.

In [ ]:
target = "abandono"
features_num = ["edad", "ingresos", "compras_12m", "visitas_web", "reclamaciones", "antiguedad_meses", "ticket_medio"]
features_cat = ["segmento"]

X = df[features_num + features_cat]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

## 4. Preprocesamiento común

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, features_num),
        ("cat", categorical_transformer, features_cat)
    ]
)

## 5. Función de evaluación de clasificación

Calculamos métricas principales:

- Accuracy.
- Precision.
- Recall.
- F1.
- ROC AUC.

In [ ]:
def evaluar_clasificador(nombre, modelo, X_train, X_test, y_train, y_test):
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)

    if hasattr(modelo, "predict_proba"):
        proba = modelo.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, proba)
    else:
        proba = None
        auc = np.nan

    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": auc
    }, pred, proba

## 6. Regresión logística

Modelo interpretable y muy usado como baseline en clasificación binaria.

In [ ]:
logit = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

res_logit, pred_logit, proba_logit = evaluar_clasificador(
    "Logistic Regression", logit, X_train, X_test, y_train, y_test
)

res_logit

## 7. Árbol de decisión y Random Forest

Los árboles permiten reglas interpretables; Random Forest suele mejorar la estabilidad y rendimiento.

In [ ]:
tree = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced"))
])

rf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

res_tree, pred_tree, proba_tree = evaluar_clasificador("Decision Tree", tree, X_train, X_test, y_train, y_test)
res_rf, pred_rf, proba_rf = evaluar_clasificador("Random Forest", rf, X_train, X_test, y_train, y_test)

pd.DataFrame([res_logit, res_tree, res_rf]).sort_values("f1", ascending=False)

## 8. Informe de clasificación

`classification_report` permite ver precision, recall y F1 por clase.

In [ ]:
print(classification_report(y_test, pred_rf, target_names=["No abandono", "Abandono"]))

## 9. Matriz de confusión

La matriz de confusión permite analizar errores concretos:

- falsos positivos,
- falsos negativos,
- verdaderos positivos,
- verdaderos negativos.

In [ ]:
cm = confusion_matrix(y_test, pred_rf)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No abandono", "Abandono"])
disp.plot()
plt.title("Matriz de confusión - Random Forest")
plt.show()

## 10. Curva ROC y AUC

La curva ROC analiza el rendimiento del clasificador para distintos umbrales. El AUC resume esa capacidad discriminativa.

In [ ]:
RocCurveDisplay.from_estimator(rf, X_test, y_test)
plt.title("Curva ROC - Random Forest")
plt.show()

print("AUC:", roc_auc_score(y_test, proba_rf))

## 11. Ajuste de umbral

El umbral por defecto es 0.5. Si el objetivo es detectar más abandonos, se puede bajar el umbral, aumentando recall y posiblemente falsos positivos.

In [ ]:
for umbral in [0.3, 0.4, 0.5, 0.6]:
    pred_umbral = (proba_rf >= umbral).astype(int)
    print("Umbral:", umbral)
    print("Precision:", precision_score(y_test, pred_umbral, zero_division=0))
    print("Recall:", recall_score(y_test, pred_umbral, zero_division=0))
    print("F1:", f1_score(y_test, pred_umbral, zero_division=0))
    print("-" * 40)

## 12. Ejercicio integrador

1. Compara los tres modelos por recall.
2. Cambia el umbral de decisión del Random Forest.
3. Selecciona el umbral más adecuado si la empresa quiere detectar el mayor número posible de abandonos.
4. Representa la matriz de confusión para ese umbral.
5. Redacta una conclusión ejecutiva.

### Pregunta de cierre

En este caso, ¿qué error es más costoso: falso positivo o falso negativo?

In [ ]:
# Espacio para el ejercicio del alumnado